# ShopAssist AI

In [2]:
"""
ShopAssist AI - Optimized Version
==================================
An intelligent laptop recommendation chatbot using Groq's Function Calling API

Key Improvements:
1. Removed unnecessary layers (intent_confirmation, dictionary_present)
2. Integrated Groq Function Calling API for structured data extraction
3. More natural and dynamic conversation flow
4. Better error handling with retry logic
5. Streamlined code with reduced complexity
6. Manual fallback mode for rate limit situations
"""


"\nShopAssist AI - Optimized Version\n==================================\nAn intelligent laptop recommendation chatbot using Groq's Function Calling API\n\nKey Improvements:\n1. Removed unnecessary layers (intent_confirmation, dictionary_present)\n2. Integrated Groq Function Calling API for structured data extraction\n3. More natural and dynamic conversation flow\n4. Better error handling with retry logic\n5. Streamlined code with reduced complexity\n6. Manual fallback mode for rate limit situations\n"

Importing required libraries for the ShopAssist

In [26]:
# import libraries
import os
#from openai import OpenAI
from groq import RateLimitError
import re 
import json
import pandas as pd
from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_exception_type



Reading the data on the fly from source

In [27]:
# Import the libraries
from IPython.display import display, HTML
# Set the display width to control the output width
pd.set_option('display.max_colwidth', None) 

# Read the dataset and read the Laptop Dataset
def load_and_parse_data():
    """Load laptop data and parse descriptions to extract structured information"""
    global df
    
    print("🔄 Loading laptop data...")
    
    # Load the CSV
    df = pd.read_csv('https://cdn.upgrad.com/uploads/production/5a1512d8-a326-4b21-8a1c-40b0a2e0082e/laptop_descriptions.csv')
    
    print(f"✅ Loaded {len(df)} laptop descriptions")
    print("🔄 Parsing descriptions to extract details...")
    
    # Extract structured data from descriptions
    parsed_data = []
    for idx, row in df.iterrows():
        description = row['laptop_description']
        data = parse_laptop_description(description)
        parsed_data.append(data)
    
    # Create dataframe from parsed data
    df_parsed = pd.DataFrame(parsed_data)
    
    # Combine with original dataframe
    df = pd.concat([df, df_parsed], axis=1)
    
    print(f"✅ Successfully parsed {len(df)} laptops")
    print(f"📊 Price range: ₹{df['Price'].min():,} - ₹{df['Price'].max():,}\n")
    
    return df

After reading the data, parse the data description and store features. 

        -> For Price, we have added RegEx and fallback method to pick any integer.
        -> RegEx applied to fetch Brand, Processor, Memory, Display.



In [28]:
def parse_laptop_description(description):
    """Extract key information from laptop description"""
    data = {'Description': description}
    
    # Extract price (look for patterns like "Priced at 55,000" or "price of 80000")
    price_match = re.search(r'(?:Priced at|price of)\s+([0-9,]+)', description, re.IGNORECASE)
    if price_match:
        price_str = price_match.group(1).replace(',', '').strip()
        if price_str:  # Check if not empty
            try:
                data['Price'] = int(price_str)
            except ValueError:
                data['Price'] = None
        else:
            data['Price'] = None
    else:
        data['Price'] = None
    
    # If price not found, try fallback: look for any 5-6 digit number at the end
    if data['Price'] is None:
        price_matches = re.findall(r'\b(\d{5,6})\b', description)
        if price_matches:
            # Take the last one (usually at the end)
            try:
                data['Price'] = int(price_matches[-1])
            except ValueError:
                data['Price'] = 50000  # Default
        else:
            data['Price'] = 50000  # Default price if not found
    
    # Extract brand and model (usually at the beginning like "The Dell Inspiron")
    brand_model_match = re.match(r'The\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)\s+([A-Z][a-zA-Z0-9\s]+?)\s+is', description)
    if brand_model_match:
        data['Brand'] = brand_model_match.group(1).strip()
        data['Model Name'] = brand_model_match.group(2).strip()
    else:
        data['Brand'] = 'Unknown'
        data['Model Name'] = 'Unknown'
    
    # Extract processor
    proc_match = re.search(r'(Intel Core i[3579]|AMD Ryzen [3579]|Apple M\d|Intel Xeon)', description)
    data['Core'] = proc_match.group(1) if proc_match else 'Unknown'
    
    # Extract RAM
    ram_match = re.search(r'(\d+GB)\s+(?:of\s+)?RAM', description, re.IGNORECASE)
    data['RAM Size'] = ram_match.group(1) if ram_match else 'Unknown'
    
    # Extract display size
    display_match = re.search(r'(\d+\.?\d*)["\']', description)
    data['Display Size'] = f"{display_match.group(1)}\"" if display_match else 'Unknown'
    
    # Extract graphics
    gpu_match = re.search(r'(NVIDIA (?:RTX|GTX|Quadro)|AMD Radeon|Intel (?:UHD|Iris|Xe)|Apple M\d)', description, re.IGNORECASE)
    data['Graphics Processor'] = gpu_match.group(1) if gpu_match else 'Unknown'
    
    return data

In [29]:
# Import the libraries
import os, json, ast
from groq import Groq
from tenacity import retry, wait_random_exponential, stop_after_attempt

In [30]:
# Read the OpenAI API key
groq_api_key = "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"

client = Groq(
    # This is the default and can be omitted
    api_key = groq_api_key #"sd-asdercc"
)

Laptop Search function which takes user input, processes and returns with top 3 recommendations

In [31]:
# LAPTOP SEARCH FUNCTION

def search_laptops(gpu_intensity, display_quality, portability, multitasking, processing_speed, budget):
    """
    Search for laptops matching user requirements
    Returns: Top 3 laptops as JSON
    """
    global df
    
    # Ensure data is loaded
    if df is None or 'Price' not in df.columns:
        print("⚠️  Data not loaded properly. Reloading...")
        load_and_parse_data()
    
    # Map requirements to laptop features
    mappings = {'low': 0, 'medium': 1, 'high': 2}
    
    print(f"🔍 Searching for laptops within budget: ₹{budget:,}")
    
    # Filter by budget
    filtered_df = df[df['Price'] <= budget].copy()
    
    print(f"📊 Found {len(filtered_df)} laptops within budget")
    
    if filtered_df.empty:
        return json.dumps({
            "error": f"No laptops found within budget of ₹{budget:,}",
            "suggestion": "Try increasing your budget or adjusting requirements",
            "min_price": int(df['Price'].min()),
            "max_price": int(df['Price'].max()),
            "laptops": []
        })
    
    # Create laptop features dictionary for each laptop
    filtered_df['laptop_feature'] = filtered_df['laptop_description'].apply(
        lambda x: extract_laptop_features(x)
    )
    
    # Score each laptop
    user_req = {
        'GPU intensity': gpu_intensity,
        'Display quality': display_quality,
        'Portability': portability,
        'Multitasking': multitasking,
        'Processing speed': processing_speed
    }
    
    filtered_df['Score'] = 0
    for index, row in filtered_df.iterrows():
        score = 0
        laptop_features = row['laptop_feature']
        
        for key, user_value in user_req.items():
            laptop_value = laptop_features.get(key, 'low')
            if mappings.get(laptop_value, 0) >= mappings.get(user_value, 0):
                score += 1
        
        filtered_df.loc[index, 'Score'] = score
    
    print(f"⭐ Top score: {filtered_df['Score'].max()}/5")
    
    # Get top 3 laptops
    top_3 = filtered_df.nlargest(3, 'Score')[[
        'Brand', 'Model Name', 'Core', 'RAM Size', 
        'Display Size', 'Graphics Processor', 
        'Price', 'Description', 'Score'
    ]]
    
    return top_3.to_json(orient='records')

In [32]:
# FUNCTION DEFINITIONS FOR GROQ FUNCTION CALLING

laptop_search_function = {
    "name": "search_laptops",
    "description": "Search for laptops based on user requirements including GPU intensity, display quality, portability, multitasking capability, processing speed, and budget",
    "parameters": {
        "type": "object",
        "properties": {
            "gpu_intensity": {
                "type": "string",
                "enum": ["low", "medium", "high"],
                "description": "Required GPU performance level"
            },
            "display_quality": {
                "type": "string",
                "enum": ["low", "medium", "high"],
                "description": "Required display quality level"
            },
            "portability": {
                "type": "string",
                "enum": ["low", "medium", "high"],
                "description": "Importance of laptop portability"
            },
            "multitasking": {
                "type": "string",
                "enum": ["low", "medium", "high"],
                "description": "Multitasking capability requirement"
            },
            "processing_speed": {
                "type": "string",
                "enum": ["low", "medium", "high"],
                "description": "Required processing speed"
            },
            "budget": {
                "type": "integer",
                "description": "Maximum budget in INR",
                "minimum": 25000
            }
        },
        "required": ["gpu_intensity", "display_quality", "portability", "multitasking", "processing_speed", "budget"]
    }
}


In [33]:
# RETRY LOGIC FOR API CALLS
from tenacity import retry, retry_if_exception_type, wait_exponential, stop_after_attempt

@retry(
retry=retry_if_exception_type(RateLimitError),
wait=wait_exponential(multiplier=2, min=10, max=120),
stop=stop_after_attempt(5)
)

def get_chat_completion_with_retry(messages, tools=None, tool_choice=None, model="llama-3.3-70b-versatile", temperature=0.7):
    """Get chat completion with automatic retry on rate limits"""
    try:
        if tools:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                tools=tools,
                tool_choice=tool_choice,
                temperature=temperature,
                max_tokens=1024
            )
        else:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=1024
            )
        return response
    except RateLimitError as e:
        print(f"⚠️  Rate limit hit. Retrying...")
        raise
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    
    


Moderation Logic

In [34]:
# MODERATION CHECK (Simplified)
def moderation_check(text):
    """
    Simplified moderation using Groq LLM
    Returns: True if content is safe, False if flagged
    """
    try:
        prompt = f"""Analyze if the following text contains inappropriate content (violence, hate speech, explicit content).
        Respond with only 'SAFE' or 'UNSAFE'.
        
        Text: {text}"""
        
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",  # Use faster model for moderation
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=10
        )
        
        result = response.choices[0].message.content.strip().upper()
        return result == "SAFE"
    except:
        return True  # Default to safe if moderation fails


Extract Laptop features from User Input using LLM

In [35]:
def extract_laptop_features(description):
    """Extract laptop features from description using LLM"""
    # Simplified feature extraction - can be enhanced
    features = {
        'GPU intensity': 'medium',
        'Display quality': 'medium',
        'Portability': 'medium',
        'Multitasking': 'medium',
        'Processing speed': 'medium'
    }
    
    desc_lower = description.lower()
    
    # GPU intensity
    if any(word in desc_lower for word in ['nvidia rtx', 'nvidia gtx', 'high-performance', 'gaming']):
        features['GPU intensity'] = 'high'
    elif any(word in desc_lower for word in ['intel uhd', 'basic']):
        features['GPU intensity'] = 'low'
    
    # Display quality
    if any(word in desc_lower for word in ['oled', '4k', '3840x2160', 'retina', 'sharp', 'vibrant']):
        features['Display quality'] = 'high'
    elif any(word in desc_lower for word in ['1366x768', 'tn display']):
        features['Display quality'] = 'low'
    
    # Portability
    if any(word in desc_lower for word in ['lightweight', 'portable', '1.', 'kg']):
        weight_match = re.search(r'(\d+\.?\d*)\s*kg', desc_lower)
        if weight_match and float(weight_match.group(1)) < 2.0:
            features['Portability'] = 'high'
    elif any(word in desc_lower for word in ['3.', 'heavy']):
        features['Portability'] = 'low'
    
    # Multitasking
    if any(word in desc_lower for word in ['32gb', '64gb', 'multitasking', 'demanding tasks']):
        features['Multitasking'] = 'high'
    elif any(word in desc_lower for word in ['8gb', 'basic']):
        features['Multitasking'] = 'low'
    
    # Processing speed
    if any(word in desc_lower for word in ['i9', 'i7', 'ryzen 7', 'ryzen 9', 'exceptional', 'powerful']):
        features['Processing speed'] = 'high'
    elif any(word in desc_lower for word in ['i3', 'basic', 'everyday']):
        features['Processing speed'] = 'low'
    
    return features


In [36]:
# CONVERSATION INITIALIZATION
def initialize_conversation():
    """Initialize conversation with improved system prompt"""
    system_message = """You are ShopAssist AI, a friendly and knowledgeable laptop expert. Your goal is to understand the user's needs through natural conversation and recommend the perfect laptop.

**Conversation Guidelines:**
1. Be conversational and friendly - avoid robotic responses
2. Ask follow-up questions naturally based on user responses
3. Don't ask all questions at once - build the conversation organically
4. When you have enough information about their needs, call the search_laptops function
5. Focus on understanding: their use case, portability needs, performance requirements, and budget

**Key Requirements to Gather:**
- **Use Case**: What will they use the laptop for? (gaming, work, study, content creation, etc.)
- **Portability**: Do they travel frequently or work from one location?
- **Performance Needs**: Do they need high processing power, good graphics, multitasking?
- **Display**: Is screen quality important for their work?
- **Budget**: What's their maximum budget? (minimum 25,000 INR)

**Important Rules:**
- Budget must be ≥ 25,000 INR
- Use the search_laptops function only when you have all 6 requirements
- Be helpful and guide users who are unsure about technical specifications
- After showing recommendations, help users compare and choose

Start with a warm greeting and ask about their laptop needs!"""

    return [{"role": "system", "content": system_message}]

Helper Function to retrieve Feature information from user conversation

In [37]:
# HELPER FUNCTION: EXTRACT REQUIREMENTS (For testing/debugging)
def extract_requirements_from_conversation(conversation_history):
    """
    Extract user requirements from conversation using LLM
    Useful for debugging or manual extraction
    """
    prompt = """Based on the conversation, extract the user's laptop requirements in JSON format:
    {
        "gpu_intensity": "low/medium/high",
        "display_quality": "low/medium/high",
        "portability": "low/medium/high",
        "multitasking": "low/medium/high",
        "processing_speed": "low/medium/high",
        "budget": <number>
    }
    
    If any requirement is unclear, use "medium" as default.
    Budget must be a number >= 25000."""
    
    messages = conversation_history + [{"role": "user", "content": prompt}]
    
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            response_format={"type": "json_object"},
            temperature=0
        )
        
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Error extracting requirements: {e}")
        return None


When the Helper Function limit is reached, we can use below manual function so that conversation is not halted. This helps user get the recommendation and reduces down time. 

In [38]:
# MANUAL FALLBACK MODE (When API is rate limited)


def manual_search_mode():
    """
    Manual mode for when API rate limits are hit
    Allows direct input of requirements
    """
    print("\n" + "="*70)
    print("🔧 MANUAL SEARCH MODE")
    print("="*70)
    print("Since the API is rate limited, let's search manually.\n")
    
    print("Please answer the following questions:\n")
    
    # Get GPU intensity
    while True:
        gpu = input("GPU intensity (low/medium/high): ").strip().lower()
        if gpu in ['low', 'medium', 'high']:
            break
        print("❌ Please enter: low, medium, or high\n")
    
    # Get display quality
    while True:
        display = input("Display quality (low/medium/high): ").strip().lower()
        if display in ['low', 'medium', 'high']:
            break
        print("❌ Please enter: low, medium, or high\n")
    
    # Get portability
    while True:
        portability = input("Portability importance (low/medium/high): ").strip().lower()
        if portability in ['low', 'medium', 'high']:
            break
        print("❌ Please enter: low, medium, or high\n")
    
    # Get multitasking
    while True:
        multitasking = input("Multitasking needs (low/medium/high): ").strip().lower()
        if multitasking in ['low', 'medium', 'high']:
            break
        print("❌ Please enter: low, medium, or high\n")
    
    # Get processing speed
    while True:
        processing = input("Processing speed (low/medium/high): ").strip().lower()
        if processing in ['low', 'medium', 'high']:
            break
        print("❌ Please enter: low, medium, or high\n")
    
    # Get budget
    while True:
        try:
            budget = int(input("Budget in INR (minimum 25000): ").strip())
            if budget >= 25000:
                break
            print("❌ Budget must be at least 25,000 INR\n")
        except ValueError:
            print("❌ Please enter a valid number\n")
    
    print(f"\n🔍 Searching for laptops...\n")
    
    # Search laptops
    try:
        results = search_laptops(gpu, display, portability, multitasking, processing, budget)
        results_data = json.loads(results)
        
        if 'error' in results_data:
            print(f"❌ {results_data['error']}")
            print(f"💡 {results_data.get('suggestion', '')}\n")
        else:
            print("✅ Top 3 Laptop Recommendations:\n")
            print("="*70)
            
            for i, laptop in enumerate(results_data, 1):
                print(f"\n{i}. {laptop['Brand']} {laptop['Model Name']}")
                print(f"   💰 Price: ₹{laptop['Price']:,}")
                print(f"   🖥️  Display: {laptop['Display Size']}")
                print(f"   ⚙️  Processor: {laptop['Core']}")
                print(f"   💾 RAM: {laptop['RAM Size']}")
                print(f"   🎮 Graphics: {laptop['Graphics Processor']}")
                print(f"   ⭐ Match Score: {laptop['Score']}/5")
            
            print("\n" + "="*70)
    except Exception as e:
        print(f"❌ Error during search: {e}\n")
    
    # Ask if they want to search again
    again = input("\nWould you like to search again? (yes/no): ").strip().lower()
    if again in ['yes', 'y']:
        manual_search_mode()
    else:
        print("\n👋 Thank you for using ShopAssist AI!\n")




Main Dialogue Management System where we capture user input and process it via Function calling API. 

In [39]:
# MAIN DIALOGUE MANAGEMENT SYSTEM (catches the input)

def dialogue_management_system():
    """
    Main conversation loop - Optimized and streamlined
    Now includes manual fallback mode for rate limit situations
    """
    print("=" * 70)
    print("🤖 ShopAssist AI - Your Intelligent Laptop Advisor")
    print("=" * 70)
    print("Type 'exit', 'quit', or 'bye' to end the conversation")
    print("Type 'manual' to use manual search mode (if API is rate limited)\n")
    
    # Initialize conversation
    conversation = initialize_conversation()
    
    # Get initial greeting
    try:
        response = get_chat_completion_with_retry(conversation)
        if not response:
            print("❌ Failed to initialize. Please try again later.")
            return
        
        assistant_message = response.choices[0].message.content
        print(f"🤖 Assistant: {assistant_message}\n")
        conversation.append({"role": "assistant", "content": assistant_message})
    except Exception as e:
        print(f"⚠️  API is currently rate limited.")
        print(f"Switching to manual search mode...\n")
        manual_search_mode()
        return
    
    # Conversation loop
    while True:
        # Get user input
        user_input = input("👤 You: ").strip()
        
        # Check for manual mode
        if user_input.lower() == 'manual':
            manual_search_mode()
            break
        
        # Check for exit commands (case-insensitive)
        if user_input.lower() in ['exit', 'quit', 'bye', 'q']:
            print("\n👋 Thank you for using ShopAssist AI! Have a great day!")
            break
        
        if not user_input:
            print("⚠️  Please enter a message or type 'exit' to quit.\n")
            continue
        
        # Moderation check
        if not moderation_check(user_input):
            print("⚠️  Your message contains inappropriate content. Please rephrase.\n")
            continue
        
        # Add user message to conversation
        conversation.append({"role": "user", "content": user_input})
        
        try:
            # Get response with function calling capability
            response = get_chat_completion_with_retry(
                messages=conversation,
                tools=[{"type": "function", "function": laptop_search_function}],
                tool_choice="auto"
            )
            
            if not response:
                print("❌ Failed to get response. Please try again.\n")
                conversation.pop()  # Remove last user message
                continue
            
            assistant_message = response.choices[0].message
            
            # Check if function was called
            if assistant_message.tool_calls:
                # Extract function call
                tool_call = assistant_message.tool_calls[0]
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"\n🔍 Searching for laptops matching your requirements via function calling...\n")
                
                # Call the function
                if function_name == "search_laptops":
                    function_response = search_laptops(**function_args)
                    
                    # Add function call and response to conversation
                    conversation.append({
                        "role": "assistant",
                        "content": None,
                        "tool_calls": [tool_call.dict()]
                    })
                    conversation.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": function_response
                    })
                    
                    # Get final response with recommendations
                    try:
                        final_response = get_chat_completion_with_retry(conversation)
                        
                        if final_response:
                            final_message = final_response.choices[0].message.content
                            print(f"🤖 Assistant: {final_message}\n")
                            conversation.append({"role": "assistant", "content": final_message})
                        else:
                            print("❌ Failed to generate recommendations.\n")
                    except Exception as e:
                        print(f"⚠️  Could not format recommendations due to rate limits.")
                        print(f"📋 Here are the raw results:\n")
                        
                        # Parse and display results manually
                        results_data = json.loads(function_response)
                        if 'error' not in results_data:
                            for i, laptop in enumerate(results_data, 1):
                                print(f"\n{i}. {laptop['Brand']} {laptop['Model Name']} - ₹{laptop['Price']:,}")
                                print(f"   {laptop['Core']}, {laptop['RAM Size']}, {laptop['Display Size']}")
                        print()
            else:
                # Regular conversation response
                content = assistant_message.content
                print(f"🤖 Assistant: {content}\n")
                conversation.append({"role": "assistant", "content": content})
        
        except RateLimitError:
            print("\n" + "="*70)
            print("⚠️  RATE LIMIT EXCEEDED")
            print("="*70)
            print("Your Groq API has hit its rate limit.")
            print("\nOptions:")
            print("1. Type 'manual' to use manual search mode (no API needed)")
            print("2. Wait and try again later")
            print("3. Type 'exit' to quit\n")
            conversation.pop()  # Remove last user message
                
        except Exception as e:
            print(f"❌ Error: {type(e).__name__}")
            print("💡 Try typing 'manual' to use manual search mode\n")
            conversation.pop()  # Remove last user message

In [40]:
# MAIN EXECUTION

if __name__ == "__main__":
    # Load data first
    load_and_parse_data()
    
    # Start the dialogue system
    dialogue_management_system()

🔄 Loading laptop data...
✅ Loaded 20 laptop descriptions
🔄 Parsing descriptions to extract details...
✅ Successfully parsed 20 laptops
📊 Price range: ₹25,000 - ₹280,000

🤖 ShopAssist AI - Your Intelligent Laptop Advisor
Type 'exit', 'quit', or 'bye' to end the conversation
Type 'manual' to use manual search mode (if API is rate limited)

🤖 Assistant: Hello! Welcome to our laptop finder service. I'm excited to help you find the perfect laptop that fits your needs. Are you in the market for a new laptop, or perhaps looking to upgrade your current one? What's the main reason you're looking for a new laptop - is it for work, study, or maybe something more creative like gaming or content creation?


🔍 Searching for laptops matching your requirements via function calling...

🔍 Searching for laptops within budget: ₹100,000
📊 Found 14 laptops within budget
⭐ Top score: 5/5


/var/folders/gw/bw9vq8556611bw9103x_tq7r0000gn/T/ipykernel_96462/1544658788.py:92: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  "tool_calls": [tool_call.dict()]


🤖 Assistant: Based on your requirements, I've found some laptops that should suit your needs. Since you're a student who needs a laptop for programming assignments in Python and Java, I've focused on laptops with good processing power, sufficient RAM, and standard displays.

Here are a few options to consider:

1. **HP EliteBook**: This laptop features an Intel Core i7 processor, 16GB of RAM, and a 14" LED display. It's lightweight, portable, and has a long-lasting battery life. Priced at 90,000, it's a great option for students who want a reliable and powerful laptop.
2. **Lenovo ThinkPad**: This laptop is equipped with a Ryzen 7 processor, 16GB of RAM, and a 14" IPS display. It's also lightweight and portable, with a backlit keyboard for comfortable typing. Priced at 60,000, it's a more affordable option that still offers strong performance.
3. **Apple MacBook Air**: If you're interested in a macOS experience, the MacBook Air is a great option. It features the Apple M1 chip, 16GB of 